In [1]:
import pandas as pd
import sys
import os
# project root to path
sys.path.append(os.path.abspath(".."))
from src.data_cleaning import *
import src.data_quality as dq
from src.config import *

print(dir(dq))

['DataQualityReport', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'check_categories', 'check_duplicates', 'check_foreign_key', 'check_missing', 'check_range', 'validate_applications', 'validate_daily', 'validate_final']


In [2]:
# Loading raw data
applications = pd.read_csv("../data/raw/applications.csv")
daily = pd.read_csv("../data/raw/daily_surveys.csv")
final = pd.read_csv("../data/raw/final_survey.csv")

print("Applications:", applications.shape)
print("Daily:", daily.shape)
print("Final:", final.shape)


Applications: (2450, 8)
Daily: (12021, 6)
Final: (1580, 5)


#### Data inspection

In [3]:
applications.head()

,student_id,year,country,gender,education_level,accepted,has_disability,disability_type
0,2024_0,2024,Kenya,non_binary,phd,1,0,none
1,2024_1,2024,India,non_binary,phd,0,0,none
2,2024_2,2024,Germany,non_binary,masters,0,1,motor
3,2024_3,2024,Peru,female,masters,0,0,none
4,2024_4,2024,Germany,female,masters,0,0,none


In [4]:
applications.info()

<class 'pandas.DataFrame'>
RangeIndex: 2450 entries, 0 to 2449
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   student_id       2450 non-null   str  
 1   year             2450 non-null   int64
 2   country          2450 non-null   str  
 3   gender           2450 non-null   str  
 4   education_level  2450 non-null   str  
 5   accepted         2450 non-null   int64
 6   has_disability   2450 non-null   int64
 7   disability_type  2450 non-null   str  
dtypes: int64(3), str(5)
memory usage: 153.3 KB


In [5]:
applications["country"].value_counts()

country
Peru              374
Kenya             351
India             346
Germany           337
UK                298
Brazil            292
USA               272
United Kingdom     65
Brasil             61
United States      31
US                 23
Name: count, dtype: int64

#### Data quality checks

In [6]:
app_report = dq.validate_applications(applications)
daily_report = dq.validate_daily(daily, applications)
final_report = dq.validate_final(final, applications)

app_report.summary()
daily_report.summary()
final_report.summary()


 Data Quality Report: Applications
----------------------------------------

⚠️ Warnings:
- 50 duplicate rows found for ['student_id']

 Data Quality Report: Daily Surveys
----------------------------------------

 Errors:
- 116 values out of range in engagement_score

 Data Quality Report: Final Survey
----------------------------------------
 No issues found


#### Cleaning application dataset

In [7]:
# standardise country names
applications = clean_countries(applications)

In [8]:
#remove duplicates
applications = remove_duplicates(applications)

In [9]:
#validate categories
applications["education_level"].value_counts()

education_level
phd          801
masters      800
undergrad    799
Name: count, dtype: int64

In [10]:
applications["education_level"] = applications["education_level"].str.lower()

#### cleaning daily survey dataset

In [11]:
#checks missing values
daily.isnull().sum()
#drop missimg rows
daily = daily.dropna(subset=["student_id", "day", "engagement_score"])
#ensure valid ranges
daily = daily[
    (daily["engagement_score"] >= 1) &
    (daily["engagement_score"] <= 5)
]

#### Cleaning final survey dataset

In [12]:
final = final.dropna(subset=["student_id", "completion"])
final["satisfaction"] = final["satisfaction"].clip(1, 5)
final["self_reported_learning"] = final["self_reported_learning"].clip(1, 5)

In [13]:
# run quality check ps on clean datasets
app_report = dq.validate_applications(applications)
daily_report = dq.validate_daily(daily, applications)
final_report = dq.validate_final(final, applications)

app_report.summary()
daily_report.summary()
final_report.summary()


 Data Quality Report: Applications
----------------------------------------
 No issues found

 Data Quality Report: Daily Surveys
----------------------------------------
 No issues found

 Data Quality Report: Final Survey
----------------------------------------
 No issues found


#### Merge dataset

In [14]:
# merge datasets

df = daily.merge(applications, on=["student_id", "year"], how="left")
df = df.merge(final, on=["student_id", "year"], how="left")

print("Merged shape:", df.shape)
df.head()

Merged shape: (11905, 15)


,student_id,year,day,attendance,engagement_score,difficulty,country,gender,education_level,accepted,has_disability,disability_type,completion,satisfaction,self_reported_learning
0,2024_256,2024,8,1,3.09,2,Kenya,female,phd,1,1,cognitive,1.0,3.48,2.88
1,2025_591,2025,2,1,2.85,2,Kenya,male,masters,1,0,none,0.0,2.33,2.69
2,2024_986,2024,3,1,4.57,4,Germany,male,undergrad,1,0,none,0.0,3.30,2.61
3,2024_1038,2024,5,1,2.86,2,India,non_binary,masters,1,0,none,1.0,3.58,3.18
4,2025_1096,2025,4,1,1.52,3,Kenya,female,phd,1,0,none,0.0,3.42,3.11


In [15]:
# basic checks

print("Missing values after merge:\n")
print(df.isnull().sum())

Missing values after merge:

student_id                  0
year                        0
day                         0
attendance                  0
engagement_score            0
difficulty                  0
country                     0
gender                      0
education_level             0
accepted                    0
has_disability              0
disability_type             0
completion                339
satisfaction              339
self_reported_learning    339
dtype: int64


In [16]:
missing_students = df["student_id"].isnull().sum()
print("Missing student IDs after merge:", missing_students)

Missing student IDs after merge: 0


In [17]:
# Handle missing final survey data
# Missing final survey data likely corresponds to students
# who did not complete the program or did not respond.
# We treat missing completion as 0 and keep a flag for later analysis
df["final_survey_missing"] = df["completion"].isna().astype(int)

df["completion"] = df["completion"].fillna(0)
df["satisfaction"] = df["satisfaction"].fillna(0)
df["self_reported_learning"] = df["self_reported_learning"].fillna(0)

df[["completion", "satisfaction", "self_reported_learning"]].isnull().sum()

completion                0
satisfaction              0
self_reported_learning    0
dtype: int64

In [18]:
# sanity checks after cleaning

df["country"].value_counts()

country
Germany    1858
Peru       1824
UK         1695
India      1687
Kenya      1667
Brazil     1646
USA        1528
Name: count, dtype: int64

In [19]:
df["disability_type"].value_counts()

disability_type
none                 8973
hearing               618
visual                584
cognitive             555
motor                 456
multiple              421
prefer_not_to_say     298
Name: count, dtype: int64

In [20]:
df["completion"].mean()

np.float64(0.6595548089038219)

In [21]:
#. save cleaned data
df.to_csv("../data/processed/merged_cleaned.csv", index=False)